# End-to-End RAG System with Document-Based Intelligent Assistant using Gradio

This notebook demonstrates how to build an **interactive document-based intelligent assistant** using a Retrieval-Augmented Generation (RAG) pipeline with **free/open-source Python libraries**.

## What this notebook covers

1. Installing required free libraries  
2. Loading open-source embedding and generation models  
3. Extracting text from PDF, TXT, and DOCX files  
4. Splitting document text into chunks  
5. Creating embeddings using Sentence Transformers  
6. Creating a FAISS vector index  
7. Retrieving relevant document chunks  
8. Generating answers using a free Hugging Face model  
9. Building an interactive Gradio application  
10. Running the app locally

## Libraries Used

| Purpose | Library |
|---|---|
| UI | Gradio |
| Embeddings | Sentence Transformers |
| Vector Search | FAISS |
| Text Generation | Hugging Face Transformers |
| PDF Reading | pypdf |
| DOCX Reading | python-docx |
| Numeric Processing | NumPy |

> This notebook does **not** require OpenAI API or any paid service.

## 1. Install Required Libraries

Run the cell below once in your notebook environment.

> If you are running this in Jupyter Notebook, restart the kernel after installation if any package was newly installed.

In [ ]:
!pip install gradio sentence-transformers faiss-cpu transformers torch pypdf python-docx numpy

## 2. Import Required Libraries

These imports support document extraction, embedding generation, vector search, answer generation, and Gradio UI creation.

In [ ]:
import os
import faiss
import numpy as np
import gradio as gr

from pypdf import PdfReader
from docx import Document as DocxDocument

from sentence_transformers import SentenceTransformer
from transformers import pipeline

## 3. Load Free Open-Source Models

We will use:

- `sentence-transformers/all-MiniLM-L6-v2` for embeddings  
- `google/flan-t5-base` for answer generation

You can switch to `google/flan-t5-small` if your system has limited RAM.

In [ ]:
# Embedding model: lightweight, free, and good for semantic search
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Free local text generation model
# For low-resource systems, replace with: google/flan-t5-small
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base"
)

## 4. Initialize Global Variables

These variables store document chunks, source metadata, and the FAISS vector index.

In [ ]:
document_chunks = []
chunk_sources = []
faiss_index = None

## 5. Document Text Extraction Functions

The assistant supports three document formats:

- PDF
- TXT
- DOCX

In [ ]:
def extract_text_from_pdf(file_path):
    """
    Extract text from a PDF file.
    """
    text = ""
    reader = PdfReader(file_path)

    for page_num, page in enumerate(reader.pages):
        page_text = page.extract_text()
        if page_text:
            text += f"

[Page {page_num + 1}]
{page_text}"

    return text


def extract_text_from_txt(file_path):
    """
    Extract text from a TXT file.
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
        return file.read()


def extract_text_from_docx(file_path):
    """
    Extract text from a DOCX file.
    """
    doc = DocxDocument(file_path)
    text = ""

    for para in doc.paragraphs:
        if para.text.strip():
            text += para.text + "
"

    return text


def extract_text(file_path):
    """
    Detect file type and extract text accordingly.
    """
    file_extension = os.path.splitext(file_path)[1].lower()

    if file_extension == ".pdf":
        return extract_text_from_pdf(file_path)

    elif file_extension == ".txt":
        return extract_text_from_txt(file_path)

    elif file_extension == ".docx":
        return extract_text_from_docx(file_path)

    else:
        return "Unsupported file format. Please upload PDF, TXT, or DOCX files."

## 6. Text Chunking

Large documents need to be split into smaller overlapping chunks before embedding.

Why chunking matters:

- Improves retrieval accuracy
- Keeps context manageable
- Avoids sending very large text to the model
- Preserves some continuity through overlap

In [ ]:
def chunk_text(text, chunk_size=700, overlap=120):
    """
    Split text into overlapping chunks.

    Parameters:
    chunk_size: Number of characters in each chunk.
    overlap: Number of overlapping characters between chunks.
    """
    chunks = []

    if not text or len(text.strip()) == 0:
        return chunks

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start = end - overlap

    return chunks

## 7. Build FAISS Vector Index

FAISS is used to store embeddings and retrieve the most relevant chunks based on semantic similarity.

We normalize embeddings and use `IndexFlatIP`, which performs inner-product search. With normalized vectors, inner product behaves like cosine similarity.

In [ ]:
def build_faiss_index(chunks):
    """
    Convert text chunks into embeddings and store them in a FAISS index.
    """
    embeddings = embedding_model.encode(chunks)
    embeddings = np.array(embeddings).astype("float32")

    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    return index

## 8. Process Uploaded Documents

This function is triggered when the user uploads documents and clicks **Process Documents** in the Gradio interface.

It performs:

1. Text extraction
2. Text chunking
3. Embedding generation
4. FAISS index creation

In [ ]:
def process_documents(files):
    """
    Process uploaded documents:
    1. Extract text
    2. Chunk text
    3. Create embeddings
    4. Build FAISS index
    """
    global document_chunks, chunk_sources, faiss_index

    document_chunks = []
    chunk_sources = []
    faiss_index = None

    if not files:
        return "Please upload at least one document."

    all_chunks = []
    all_sources = []

    for file in files:
        file_path = file.name
        file_name = os.path.basename(file_path)

        extracted_text = extract_text(file_path)

        if extracted_text.startswith("Unsupported"):
            continue

        chunks = chunk_text(extracted_text)

        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_sources.append(f"{file_name} | Chunk {i + 1}")

    if len(all_chunks) == 0:
        return "No readable text found in the uploaded documents."

    document_chunks = all_chunks
    chunk_sources = all_sources

    faiss_index = build_faiss_index(document_chunks)

    return f"""
Documents processed successfully.

Total uploaded files: {len(files)}
Total text chunks created: {len(document_chunks)}
Vector index: FAISS index created successfully

You can now ask questions from the uploaded documents.
"""

## 9. Retrieve Relevant Chunks

For every user question:

1. Convert the question into an embedding
2. Search the FAISS index
3. Return top-k most relevant chunks

In [ ]:
def retrieve_relevant_chunks(query, top_k=4):
    """
    Retrieve top-k most relevant chunks for a user query.
    """
    global document_chunks, chunk_sources, faiss_index

    if faiss_index is None:
        return []

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(query_embedding, top_k)

    retrieved_chunks = []

    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            retrieved_chunks.append({
                "chunk": document_chunks[idx],
                "source": chunk_sources[idx],
                "score": float(score)
            })

    return retrieved_chunks

## 10. Generate Answer using Retrieved Context

The generation model receives:

- Retrieved context
- User question
- Instruction to answer only from the provided context

This helps reduce hallucination.

In [ ]:
def generate_answer(query, retrieved_chunks):
    """
    Generate final answer using a local free LLM model.
    """
    if not retrieved_chunks:
        return "I could not find relevant information in the uploaded documents."

    context = "

".join(
        [f"Source: {item['source']}
Content: {item['chunk']}" for item in retrieved_chunks]
    )

    prompt = f"""
You are a helpful document-based intelligent assistant.

Use only the information provided in the context below to answer the question.
If the answer is not available in the context, say:
"I could not find this information in the uploaded documents."

Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=300,
        do_sample=False
    )

    answer = response[0]["generated_text"]

    return answer

## 11. Main Question-Answer Function

This function connects retrieval and generation.

It returns:

- Final answer
- Sources used
- Similarity scores

In [ ]:
def ask_question(query, top_k):
    """
    Main function called by Gradio when user asks a question.
    """
    if faiss_index is None:
        return "Please upload and process documents first."

    if not query.strip():
        return "Please enter a valid question."

    retrieved_chunks = retrieve_relevant_chunks(query, top_k=top_k)

    answer = generate_answer(query, retrieved_chunks)

    source_text = "

Sources Used:
"

    for i, item in enumerate(retrieved_chunks, 1):
        source_text += f"
{i}. {item['source']} | Similarity Score: {item['score']:.4f}"

    final_response = answer + "
" + source_text

    return final_response

## 12. Build Gradio Interactive UI

The UI has two main areas:

- Left panel: upload and process documents
- Right panel: ask questions and view answers

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # Document-Based Intelligent Assistant using RAG

        Upload your documents and ask questions from them.

        This app uses:
        - Sentence Transformers for embeddings
        - FAISS for vector search
        - FLAN-T5 for answer generation
        - Gradio for interactive UI
        - No paid API required
        """
    )

    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown("## Step 1: Upload Documents")

            file_input = gr.File(
                label="Upload PDF, TXT, or DOCX files",
                file_count="multiple",
                file_types=[".pdf", ".txt", ".docx"]
            )

            process_button = gr.Button("Process Documents")

            processing_status = gr.Textbox(
                label="Processing Status",
                lines=8
            )

            top_k_slider = gr.Slider(
                minimum=1,
                maximum=8,
                value=4,
                step=1,
                label="Number of Relevant Chunks to Retrieve"
            )

        with gr.Column(scale=2):
            gr.Markdown("## Step 2: Ask Questions")

            question_input = gr.Textbox(
                label="Ask a question from your documents",
                placeholder="Example: What are the key points mentioned in the document?",
                lines=3
            )

            ask_button = gr.Button("Ask Assistant")

            answer_output = gr.Textbox(
                label="Assistant Response",
                lines=18
            )

    process_button.click(
        fn=process_documents,
        inputs=file_input,
        outputs=processing_status
    )

    ask_button.click(
        fn=ask_question,
        inputs=[question_input, top_k_slider],
        outputs=answer_output
    )

## 13. Launch the App

Run this cell to start the Gradio app.

After running, Gradio will show a local URL such as:

```text
http://127.0.0.1:7860
```

Open the URL in your browser.

In [ ]:
# Launch the app
# Set share=True only if you want a temporary public link.
demo.launch()

## 14. Optional: Save as a Standalone Python App

If you want to run this outside Jupyter Notebook, use the code below to create `rag_gradio_app.py`.

Then run:

```bash
python rag_gradio_app.py
```

In [ ]:
app_code = '\n# End-to-End RAG System with Document-Based Intelligent Assistant\n# Free Libraries + Gradio UI\n\nimport os\nimport faiss\nimport numpy as np\nimport gradio as gr\n\nfrom pypdf import PdfReader\nfrom docx import Document as DocxDocument\nfrom sentence_transformers import SentenceTransformer\nfrom transformers import pipeline\n\nembedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")\n\ngenerator = pipeline(\n    "text2text-generation",\n    model="google/flan-t5-base",\n    tokenizer="google/flan-t5-base"\n)\n\ndocument_chunks = []\nchunk_sources = []\nfaiss_index = None\n\n\ndef extract_text_from_pdf(file_path):\n    text = ""\n    reader = PdfReader(file_path)\n    for page_num, page in enumerate(reader.pages):\n        page_text = page.extract_text()\n        if page_text:\n            text += f"\\n\\n[Page {page_num + 1}]\\n{page_text}"\n    return text\n\n\ndef extract_text_from_txt(file_path):\n    with open(file_path, "r", encoding="utf-8", errors="ignore") as file:\n        return file.read()\n\n\ndef extract_text_from_docx(file_path):\n    doc = DocxDocument(file_path)\n    text = ""\n    for para in doc.paragraphs:\n        if para.text.strip():\n            text += para.text + "\\n"\n    return text\n\n\ndef extract_text(file_path):\n    file_extension = os.path.splitext(file_path)[1].lower()\n    if file_extension == ".pdf":\n        return extract_text_from_pdf(file_path)\n    elif file_extension == ".txt":\n        return extract_text_from_txt(file_path)\n    elif file_extension == ".docx":\n        return extract_text_from_docx(file_path)\n    else:\n        return "Unsupported file format. Please upload PDF, TXT, or DOCX files."\n\n\ndef chunk_text(text, chunk_size=700, overlap=120):\n    chunks = []\n    if not text or len(text.strip()) == 0:\n        return chunks\n    start = 0\n    while start < len(text):\n        end = start + chunk_size\n        chunk = text[start:end]\n        if chunk.strip():\n            chunks.append(chunk.strip())\n        start = end - overlap\n    return chunks\n\n\ndef build_faiss_index(chunks):\n    embeddings = embedding_model.encode(chunks)\n    embeddings = np.array(embeddings).astype("float32")\n    faiss.normalize_L2(embeddings)\n    dimension = embeddings.shape[1]\n    index = faiss.IndexFlatIP(dimension)\n    index.add(embeddings)\n    return index\n\n\ndef process_documents(files):\n    global document_chunks, chunk_sources, faiss_index\n    document_chunks = []\n    chunk_sources = []\n    faiss_index = None\n\n    if not files:\n        return "Please upload at least one document."\n\n    all_chunks = []\n    all_sources = []\n\n    for file in files:\n        file_path = file.name\n        file_name = os.path.basename(file_path)\n        extracted_text = extract_text(file_path)\n        if extracted_text.startswith("Unsupported"):\n            continue\n        chunks = chunk_text(extracted_text)\n        for i, chunk in enumerate(chunks):\n            all_chunks.append(chunk)\n            all_sources.append(f"{file_name} | Chunk {i + 1}")\n\n    if len(all_chunks) == 0:\n        return "No readable text found in the uploaded documents."\n\n    document_chunks = all_chunks\n    chunk_sources = all_sources\n    faiss_index = build_faiss_index(document_chunks)\n\n    return f"""\nDocuments processed successfully.\n\nTotal uploaded files: {len(files)}\nTotal text chunks created: {len(document_chunks)}\nVector index: FAISS index created successfully\n\nYou can now ask questions from the uploaded documents.\n"""\n\n\ndef retrieve_relevant_chunks(query, top_k=4):\n    global document_chunks, chunk_sources, faiss_index\n    if faiss_index is None:\n        return []\n    query_embedding = embedding_model.encode([query])\n    query_embedding = np.array(query_embedding).astype("float32")\n    faiss.normalize_L2(query_embedding)\n    scores, indices = faiss_index.search(query_embedding, top_k)\n    retrieved_chunks = []\n    for score, idx in zip(scores[0], indices[0]):\n        if idx != -1:\n            retrieved_chunks.append({\n                "chunk": document_chunks[idx],\n                "source": chunk_sources[idx],\n                "score": float(score)\n            })\n    return retrieved_chunks\n\n\ndef generate_answer(query, retrieved_chunks):\n    if not retrieved_chunks:\n        return "I could not find relevant information in the uploaded documents."\n\n    context = "\\n\\n".join(\n        [f"Source: {item[\'source\']}\\nContent: {item[\'chunk\']}" for item in retrieved_chunks]\n    )\n\n    prompt = f"""\nYou are a helpful document-based intelligent assistant.\n\nUse only the information provided in the context below to answer the question.\nIf the answer is not available in the context, say:\n"I could not find this information in the uploaded documents."\n\nContext:\n{context}\n\nQuestion:\n{query}\n\nAnswer:\n"""\n    response = generator(prompt, max_new_tokens=300, do_sample=False)\n    return response[0]["generated_text"]\n\n\ndef ask_question(query, top_k):\n    if faiss_index is None:\n        return "Please upload and process documents first."\n    if not query.strip():\n        return "Please enter a valid question."\n    retrieved_chunks = retrieve_relevant_chunks(query, top_k=top_k)\n    answer = generate_answer(query, retrieved_chunks)\n    source_text = "\\n\\nSources Used:\\n"\n    for i, item in enumerate(retrieved_chunks, 1):\n        source_text += f"\\n{i}. {item[\'source\']} | Similarity Score: {item[\'score\']:.4f}"\n    return answer + "\\n" + source_text\n\n\nwith gr.Blocks(theme=gr.themes.Soft()) as demo:\n    gr.Markdown(\n        """\n        # Document-Based Intelligent Assistant using RAG\n\n        Upload your documents and ask questions from them.\n\n        This app uses free/open-source libraries only:\n        - Sentence Transformers\n        - FAISS\n        - FLAN-T5\n        - Gradio\n        """\n    )\n\n    with gr.Row():\n        with gr.Column(scale=1):\n            gr.Markdown("## Step 1: Upload Documents")\n            file_input = gr.File(\n                label="Upload PDF, TXT, or DOCX files",\n                file_count="multiple",\n                file_types=[".pdf", ".txt", ".docx"]\n            )\n            process_button = gr.Button("Process Documents")\n            processing_status = gr.Textbox(label="Processing Status", lines=8)\n            top_k_slider = gr.Slider(\n                minimum=1,\n                maximum=8,\n                value=4,\n                step=1,\n                label="Number of Relevant Chunks to Retrieve"\n            )\n\n        with gr.Column(scale=2):\n            gr.Markdown("## Step 2: Ask Questions")\n            question_input = gr.Textbox(\n                label="Ask a question from your documents",\n                placeholder="Example: What are the key points mentioned in the document?",\n                lines=3\n            )\n            ask_button = gr.Button("Ask Assistant")\n            answer_output = gr.Textbox(label="Assistant Response", lines=18)\n\n    process_button.click(fn=process_documents, inputs=file_input, outputs=processing_status)\n    ask_button.click(fn=ask_question, inputs=[question_input, top_k_slider], outputs=answer_output)\n\n\nif __name__ == "__main__":\n    demo.launch()\n'

with open("rag_gradio_app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("rag_gradio_app.py created successfully.")
print("Run it using: python rag_gradio_app.py")

## 15. Suggested Training Lab Activities

Ask participants to try the following:

1. Upload one PDF and ask factual questions.
2. Change `top_k` from 2 to 6 and compare answer quality.
3. Replace `google/flan-t5-base` with `google/flan-t5-small` for faster execution.
4. Try PDF vs DOCX vs TXT and compare extraction quality.
5. Modify `chunk_size` and `overlap` to understand impact on retrieval.

## Summary

This notebook provides a complete free-library RAG pipeline:

```text
Documents → Text Extraction → Chunking → Embeddings → FAISS → Retrieval → Generation → Gradio UI
```